# EUR Threshold Fix

At P ≤ 1×10⁻⁴ the mean P is 1.35×10⁻³ for EUR-EUR and 1.35×10⁻³ for EUR-AFR, but a mean can't exceed the threshold if every variant passes it. Some weighted variants must have P above the threshold, so we check the P column in the EUR files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

thresholds = {"P5e-08":5e-8,"P1e-05":1e-5,"P0.0001":1e-4,
              "P0.001":1e-3,"P0.01":1e-2,"P0.05":5e-2}

pipelines = {
    "EUR-AFR": "/content/drive/MyDrive/PRS_T2D_E-A/results/weights",
    "EUR-EUR": "/content/drive/MyDrive/PRS_T2D_E-E/results/weights",
}

rows = []
for pipe, base in pipelines.items():
    for t, cut in thresholds.items():
        f = Path(base) / t / "MVP_T2D_EUR_weights.tsv"
        df = pd.read_csv(f, sep="\t")
        p = pd.to_numeric(df["P"], errors="coerce")
        rows.append({
            "Pipeline": pipe, "Threshold": t, "Cutoff": cut,
            "Rows": len(df),
            "Rows_above_cutoff": (p > cut).sum(),
            "Max_P": p.max(),
            "Missing_P": p.isna().sum(),
            "Zero_P": (p == 0).sum(),
        })

check = pd.DataFrame(rows)
display(check)

# Show a few offending rows from the worst file
worst = check.sort_values("Rows_above_cutoff", ascending=False).iloc[0]
f = Path(pipelines[worst["Pipeline"]]) / worst["Threshold"] / "MVP_T2D_EUR_weights.tsv"
df = pd.read_csv(f, sep="\t")
display(df[df["P"] > worst["Cutoff"]].sort_values("P", ascending=False).head(10))

Mounted at /content/drive


,Pipeline,Threshold,Cutoff,Rows,Rows_above_cutoff,Max_P,Missing_P,Zero_P
0,EUR-AFR,P5e-08,5.000000e-08,4944,27,0.6966,0,0
1,EUR-AFR,P1e-05,1.000000e-05,16558,73,0.9770,0,0
2,EUR-AFR,P0.0001,1.000000e-04,37511,139,0.9770,0,0
3,EUR-AFR,P0.001,1.000000e-03,107335,335,0.9983,0,0
4,EUR-AFR,P0.01,1.000000e-02,364262,886,0.9994,0,0
5,EUR-AFR,P0.05,5.000000e-02,920240,1690,0.9999,0,0
6,EUR-EUR,P5e-08,5.000000e-08,2063,9,0.7617,0,0
7,EUR-EUR,P1e-05,1.000000e-05,9216,46,0.9721,0,0
8,EUR-EUR,P0.0001,1.000000e-04,25465,93,0.9729,0,0
9,EUR-EUR,P0.001,1.000000e-03,83925,266,0.9980,0,0


,SNP,CHR,BP,A1,A2,EAF,BETA,SE,P,N
257178,rs180919580,4,109754209.0,C,G,0.9937,0.0000,0.055184,0.9999,432648
879746,rs529294702,20,8515448.0,G,A,0.9997,-0.0002,0.200088,0.9994,432648
514959,rs10738725,9,25603441.0,T,C,0.2294,0.0000,0.008493,0.9989,432648
161236,rs6806266,3,15551207.0,T,A,0.3578,0.0000,0.007680,0.9988,432648
78365,rs7595148,2,10531427.0,C,A,0.8637,0.0000,0.009133,0.9983,432648
332227,rs10477024,5,152633517.0,A,C,0.7509,0.0000,0.007524,0.9980,432648
858911,rs74174352,19,5871718.0,T,C,0.9928,-0.0001,0.038360,0.9979,432648
352631,rs143171597,6,27624160.0,C,G,0.9944,0.0000,0.041874,0.9961,432648
560731,rs4838434,10,48986166.0,T,A,0.9880,-0.0002,0.030804,0.9959,432648
281343,rs77925432,4,186859601.0,G,A,0.9986,0.0010,0.133472,0.9953,432648


*In every EUR file, 0.2–0.6% of weighted variants have P above the cutoff (for example 93 of 25,465 in EUR-EUR at P ≤ 1×10⁻⁴).*

In [ ]:
cut = 1e-4
w = pd.read_csv("/content/drive/MyDrive/PRS_T2D_E-E/results/weights/P0.0001/MVP_T2D_EUR_weights.tsv", sep="\t")
c = pd.read_csv("/content/drive/MyDrive/PRS_T2D_E-E/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_P0.0001.tsv", sep="\t")

bad = w[w["P"] > cut]
print("Bad rows:", len(bad))
print("Bad SNP IDs found in candidate file:", bad["SNP"].isin(c["SNP"]).sum())
print("Bad SNP IDs duplicated in candidate file:", c["SNP"].isin(bad["SNP"]).sum() - bad["SNP"].isin(c["SNP"]).sum())

# Did the bad rows come from the GWAS with a different allele pair?
m = bad.merge(c, on="SNP", how="left", suffixes=("_w","_c"))
print(m[["SNP","A1_w","A2_w","P_w","A1_c","A2_c","P_c"]].head(10))

print("Rows kept after filtering P <= cutoff:", (w["P"] <= cut).sum(), "of", len(w))

Bad rows: 93
Bad SNP IDs found in candidate file: 93
Bad SNP IDs duplicated in candidate file: 1
           SNP A1_w A2_w       P_w A1_c A2_c           P_c
0   rs11122090    C    A  0.000266    C    T  3.287000e-05
1    rs1630312    C    G  0.054910    C    T  4.843000e-19
2  rs111842135    C    A  0.192200    C    T  3.251000e-05
3     rs951416    T    A  0.018630    T    C  9.191000e-07
4  rs140879042    A    C  0.058260    A    G  3.666000e-06
5   rs61359279    C    A  0.567300    C    T  1.986000e-06
6   rs66498268    C    G  0.765900    C    T  2.220000e-05
7   rs16841087    G    A  0.805300    G    C  7.797000e-05
8  rs113004050    T    C  0.464100    T    G  4.018000e-05
9   rs78023211    C    A  0.641800    C    T  3.006000e-05
Rows kept after filtering P <= cutoff: 25372 of 25465


*All 93 bad rows are multi-allelic sites. The same rsID exists with more than one alternate allele, and the weight file picked the wrong one. Look at A2: the candidate file has C/T with P = 3×10⁻⁵, while the weight row has C/A with P = 0.0003. These are different variants sharing an ID. The bad rows carry the wrong BETA, EAF and P, so they are real index SNPs with wrong weights.*

In [ ]:
import pandas as pd
from pathlib import Path

cut_map = {"P5e-08":5e-8,"P1e-05":1e-5,"P0.0001":1e-4,"P0.001":1e-3,"P0.01":1e-2,"P0.05":5e-2}
cand_names = {"EUR-EUR": "MVP_T2D_Phe_250_2_EUR_{t}.tsv",
              "EUR-AFR": "MVP_T2D_Phe_250_2_EUR_{t}.tsv"}
roots = {"EUR-EUR":"/content/drive/MyDrive/PRS_T2D_E-E", "EUR-AFR":"/content/drive/MyDrive/PRS_T2D_E-A"}

out = []
for pipe, root in roots.items():
    for t in cut_map:
        w = pd.read_csv(f"{root}/results/weights/{t}/MVP_T2D_EUR_weights.tsv", sep="\t")
        c = pd.read_csv(f"{root}/results/candidate_snps/{cand_names[pipe].format(t=t)}", sep="\t")
        # Rebuild: keep candidate rows for the index SNPs; if an ID has 2 alleles, keep lowest P
        c = c.sort_values("P").drop_duplicates("SNP")
        fixed = c[c["SNP"].isin(w["SNP"])].copy()
        # Audit: how many original weight rows disagree with the candidate row?
        m = w.merge(fixed, on="SNP", suffixes=("_w","_c"))
        mismatch = ((m["A2_w"]!=m["A2_c"]) | (m["BETA_w"]!=m["BETA_c"])).sum()
        out.append({"Pipeline":pipe,"Threshold":t,"Original":len(w),"Rebuilt":len(fixed),
                    "Mismatched_rows":mismatch,"Max_P_rebuilt":fixed["P"].max()})
        # save corrected file (new name, don't overwrite)
        fixed.to_csv(f"{root}/results/weights/{t}/MVP_T2D_EUR_weights_FIXED.tsv", sep="\t", index=False)

display(pd.DataFrame(out))

,Pipeline,Threshold,Original,Rebuilt,Mismatched_rows,Max_P_rebuilt
0,EUR-EUR,P5e-08,2063,2063,16,4.978000e-08
1,EUR-EUR,P1e-05,9216,9216,58,1.000000e-05
2,EUR-EUR,P0.0001,25465,25465,115,1.000000e-04
3,EUR-EUR,P0.001,83925,83925,305,1.000000e-03
4,EUR-EUR,P0.01,319256,319256,871,1.000000e-02
5,EUR-EUR,P0.05,842495,842495,1856,5.000000e-02
6,EUR-AFR,P5e-08,4944,4944,32,4.998000e-08
7,EUR-AFR,P1e-05,16558,16558,83,1.000000e-05
8,EUR-AFR,P0.0001,37511,37511,155,1.000000e-04
9,EUR-AFR,P0.001,107335,107335,363,1.000000e-03


### Checking for AFRR

In [ ]:

import pandas as pd

cut_map = {"P5e-08":5e-8,"P1e-05":1e-5,"P0.0001":1e-4,
           "P0.001":1e-3,"P0.01":1e-2,"P0.05":5e-2}

afr_cfg = {
    "AFR-AFR": {"root": "/content/drive/MyDrive/PRS_T2D",
                "weights": "MVP_T2D_AFR_weights.tsv"},
    "AFR-EUR": {"root": "/content/drive/MyDrive/PRS_T2D_A-E",
                "weights": "T2D_PheCode250_2_AFR_weights.tsv"},
}

out = []
for pipe, cfg in afr_cfg.items():
    for t in cut_map:
        w = pd.read_csv(f"{cfg['root']}/results/weights/{t}/{cfg['weights']}", sep="\t")
        c = pd.read_csv(f"{cfg['root']}/results/candidate_snps/T2D_PheCode250_2_{t}.tsv", sep="\t")
        c = c.sort_values("P").drop_duplicates("SNP")
        fixed = c[c["SNP"].isin(w["SNP"])].copy()

        m = w.merge(fixed, on="SNP", suffixes=("_w", "_c"))
        mismatch = ((m["A2_w"] != m["A2_c"]) | (m["BETA_w"] != m["BETA_c"])).sum()

        fixed_name = cfg["weights"].replace(".tsv", "_FIXED.tsv")
        fixed.to_csv(f"{cfg['root']}/results/weights/{t}/{fixed_name}", sep="\t", index=False)

        out.append({"Pipeline": pipe, "Threshold": t,
                    "Original": len(w), "Rebuilt": len(fixed),
                    "Mismatched_rows": mismatch,
                    "Rows_above_cutoff_original": (w["P"] > cut_map[t]).sum(),
                    "Max_P_rebuilt": fixed["P"].max()})

afr_audit = pd.DataFrame(out)
print(afr_audit.to_string(index=False))

Pipeline Threshold  Original  Rebuilt  Mismatched_rows  Rows_above_cutoff_original  Max_P_rebuilt
 AFR-AFR    P5e-08      1710     1710                0                           0   4.944000e-08
 AFR-AFR    P1e-05      4305     4305                0                           0   9.961000e-06
 AFR-AFR   P0.0001      7666     7666                0                           0   1.000000e-04
 AFR-AFR    P0.001     19691    19691                0                           0   1.000000e-03
 AFR-AFR     P0.01     90187    90187                0                           0   1.000000e-02
 AFR-AFR     P0.05    357804   357804                0                           0   5.000000e-02
 AFR-EUR    P5e-08       633      633                0                           0   4.930000e-08
 AFR-EUR    P1e-05      1636     1636                0                           0   9.943000e-06
 AFR-EUR   P0.0001      3302     3302                0                           0   9.992000e-05
 AFR-EUR    P0.001  

*No error found*

### Confirming the fix worked!

In [ ]:
import pandas as pd

cut_map = {"P5e-08":5e-8,"P1e-05":1e-5,"P0.0001":1e-4,
           "P0.001":1e-3,"P0.01":1e-2,"P0.05":5e-2}
base = "/content/drive/MyDrive"
cfg = {
 "AFR-AFR": (f"{base}/PRS_T2D",     "MVP_T2D_AFR_weights"),
 "AFR-EUR": (f"{base}/PRS_T2D_A-E", "T2D_PheCode250_2_AFR_weights"),
 "EUR-AFR": (f"{base}/PRS_T2D_E-A", "MVP_T2D_EUR_weights"),
 "EUR-EUR": (f"{base}/PRS_T2D_E-E", "MVP_T2D_EUR_weights"),
}

def stats(df, cut):
    return {"N": len(df), "Dup_SNPs": df["SNP"].duplicated().sum(),
            "Above_cut": (df["P"] > cut).sum(), "Max_P": df["P"].max(),
            "Mean_P": df["P"].mean(), "Med_absBETA": df["BETA"].abs().median(),
            "Med_EAF": df["EAF"].median(), "Pct_pos": (df["BETA"] > 0).mean()*100}

rows = []
for pipe, (root, stem) in cfg.items():
    for t, cut in cut_map.items():
        o = pd.read_csv(f"{root}/results/weights/{t}/{stem}.tsv", sep="\t")
        f = pd.read_csv(f"{root}/results/weights/{t}/{stem}_FIXED.tsv", sep="\t")
        so, sf = stats(o, cut), stats(f, cut)
        rows.append({"Pipe": pipe, "Thr": t,
            "N_orig": so["N"], "N_fixed": sf["N"],
            "Above_orig": so["Above_cut"], "Above_fixed": sf["Above_cut"],
            "Dups_fixed": sf["Dup_SNPs"],
            "MaxP_fixed": sf["Max_P"], "MeanP_orig": so["Mean_P"], "MeanP_fixed": sf["Mean_P"],
            "MedAbsB_orig": so["Med_absBETA"], "MedAbsB_fixed": sf["Med_absBETA"],
            "MedEAF_orig": so["Med_EAF"], "MedEAF_fixed": sf["Med_EAF"],
            "Pos%_orig": so["Pct_pos"], "Pos%_fixed": sf["Pct_pos"]})

report = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 50, "display.float_format", "{:.4g}".format)
print(report.to_string(index=False))

# Same-SNP-set check: fixed files must contain the same SNPs as the originals
chk = []
for pipe, (root, stem) in cfg.items():
    for t in cut_map:
        o = pd.read_csv(f"{root}/results/weights/{t}/{stem}.tsv", sep="\t", usecols=["SNP"])
        f = pd.read_csv(f"{root}/results/weights/{t}/{stem}_FIXED.tsv", sep="\t", usecols=["SNP"])
        chk.append({"Pipe": pipe, "Thr": t, "Same_SNP_set": set(o["SNP"]) == set(f["SNP"])})
print(pd.DataFrame(chk)["Same_SNP_set"].all(), "<- all fixed files contain the same SNPs as the originals")

   Pipe     Thr  N_orig  N_fixed  Above_orig  Above_fixed  Dups_fixed  MaxP_fixed  MeanP_orig  MeanP_fixed  MedAbsB_orig  MedAbsB_fixed  MedEAF_orig  MedEAF_fixed  Pos%_orig  Pos%_fixed
AFR-AFR  P5e-08    1710     1710           0            0           0   4.944e-08    5.46e-09     5.46e-09       0.01327        0.01327       0.4615        0.4615       46.9        46.9
AFR-AFR  P1e-05    4305     4305           0            0           0   9.961e-06    1.71e-06     1.71e-06       0.01076        0.01076         0.44          0.44      49.13       49.13
AFR-AFR P0.0001    7666     7666           0            0           0      0.0001   2.124e-05    2.124e-05        0.0101         0.0101       0.4299        0.4299      50.03       50.03
AFR-AFR  P0.001   19691    19691           0            0           0       0.001   0.0003117    0.0003117       0.01097        0.01097       0.3868        0.3868      51.07       51.07
AFR-AFR   P0.01   90187    90187           0            0           0 